# Feature embeddings with TabFORGE

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SilenceX12138/TabFORGE/blob/master/docs/tutorial/notebooks/embedding/01_feature_embeddings.ipynb)

> **Colab setup:** Open this notebook with the badge, then follow the [tutorial setup guide](https://silencex12138.github.io/TabFORGE/#/tutorials.md) to install `tabforge-ml[tutorials]` from PyPI in this runtime. Select a GPU for pretrained workloads.

## 1. Overview

This tutorial fits an unsupervised embedder on German Credit, extracts encoder and Denoising-aligned Latent Decoder
layers with several pooling strategies, and evaluates a frozen linear probe. Labels are used only by
the probe, never while fitting TabFORGE.


In [ ]:
%pip install "tabforge-ml[tutorials]"

## 2. Imports and setup

The default is a fast deterministic mock backend so the notebook executes offline. Set
`TABFORGE_TUTORIAL_MODEL_PATH=auto` to use the pinned pretrained Structure-aware Feature Encoder and backbone.

In [ ]:
import os
import tempfile
from pathlib import Path

from tabforge import (
    MODEL_ID,
    TabFORGEArchitectureConfig,
    TabFORGEEmbeddingConfig,
    TabFORGERuntimeConfig,
    TabFORGETrainingConfig,
)

RANDOM_STATE = 7
MAX_TRAIN_ROWS = int(os.environ.get("TABFORGE_TUTORIAL_MAX_ROWS", "256"))
MAX_STEPS = int(os.environ.get("TABFORGE_TUTORIAL_MAX_STEPS", "10"))
DEVICE = os.environ.get("TABFORGE_TUTORIAL_DEVICE", "auto")
MODEL_PATH = os.environ.get("TABFORGE_TUTORIAL_MODEL_PATH", "mock")
CHECKPOINT = None if MODEL_PATH == "mock" else "pretrained"

assert MODEL_ID == "tabforge-v1"

## 3. Prepare a real-world dataset

`TabularDataset` downloads German Credit from OpenML, keeps the mixed numerical/categorical schema,
and performs a reproducible stratified split. The tutorial caps the training rows to keep execution
short; increase `TABFORGE_TUTORIAL_MAX_ROWS` for a larger run.

In [ ]:
from tabcamel.data.dataset import TabularDataset

dataset = TabularDataset("credit-g", task_type="classification")
split = dataset.split("stratified", train_size=0.8, random_state=RANDOM_STATE)
train_set = split["train_set"]
test_set = split["test_set"]

X_train = train_set.X_df.iloc[:MAX_TRAIN_ROWS].reset_index(drop=True)
y_train = train_set.y_s.iloc[:MAX_TRAIN_ROWS].reset_index(drop=True)
X_test = test_set.X_df.reset_index(drop=True)
y_test = test_set.y_s.reset_index(drop=True)
categorical_features = tuple(dataset.categorical_feature_list)

print({"train": X_train.shape, "test": X_test.shape, "target": y_train.name})

## 4. Configure TabFORGE

The constructor controls model fitting. Embedding source, layer, and pooling stay at extraction time.

In [ ]:
from tabforge import TabFORGEEmbedder

embedder = TabFORGEEmbedder(
    task="unsupervision",
    categorical_features=categorical_features,
    embedding_config=TabFORGEEmbeddingConfig(model_path=MODEL_PATH, n_folds=0 if MODEL_PATH == "mock" else 2),
    architecture_config=(
        TabFORGEArchitectureConfig(
            embedding_dimension=8 if MODEL_PATH == "mock" else None,
            decoder_layers=1, decoder_heads=2, decoder_ffn_factor=2,
            denoiser_layers=1, denoiser_heads=2, denoiser_ffn_factor=2,
        )
        if MODEL_PATH == "mock"
        else None
    ),
    training_config=TabFORGETrainingConfig(max_steps=MAX_STEPS, batch_size=128),
    runtime_config=TabFORGERuntimeConfig(device=DEVICE),
    random_state=RANDOM_STATE,
)

## 5. Fit

Fitting prepares the table schema, embedding context, Denoising-aligned Latent Decoder, and diffusion backbone.

In [ ]:
embedder.fit(X_train, checkpoint=CHECKPOINT)

## 6. Generate embeddings

`transform(X)` remains the default Structure-aware Feature Encoder token grid. Call-time arguments select an intermediate
encoder or Denoising-aligned Latent Decoder layer and optionally produce one vector per row.

In [ ]:
query = X_test.iloc[:32]
token_grid = embedder.transform(query)
encoder_layer_0 = embedder.transform(query, source="encoder", layer=0)
decoder_layer_0 = embedder.transform(query, source="decoder", layer=0)
mean_pooled = embedder.transform(query, pooling="mean")
max_pooled = embedder.transform(query, pooling="max")
flattened = embedder.transform(query, pooling="flatten")

## 7. Inspect results

No pooling preserves token identity. Mean and max pooling produce compact vectors; flattening preserves
every token coordinate. The legacy pooling helpers remain available for older code.

In [ ]:
print({
    "token_grid": token_grid.shape,
    "mean_pooled": mean_pooled.shape,
    "max_pooled": max_pooled.shape,
    "flattened": flattened.shape,
    "encoder_layer_0": encoder_layer_0.shape,
    "decoder_layer_0": decoder_layer_0.shape,
})

## 8. Frozen linear probe

A linear probe measures embedding quality without updating TabFORGE. The same frozen embedder
transforms train and test rows; only logistic regression sees the labels.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score

Z_train = embedder.transform(X_train, pooling="mean")
Z_test = embedder.transform(X_test, pooling="mean")
probe = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE).fit(Z_train, y_train)
probe_predictions = probe.predict(Z_test)
print(
    {
        "accuracy": accuracy_score(y_test, probe_predictions),
        "balanced_accuracy": balanced_accuracy_score(y_test, probe_predictions),
    }
)

## 9. Save, load, and sklearn interoperability

The embedding checkpoint retains the fitted external-provider context, without encoder checkpoint bytes.

In [ ]:
from sklearn.base import clone

assert clone(embedder).get_params()["task"] == "unsupervision"
with tempfile.TemporaryDirectory() as directory:
    checkpoint_path = embedder.save_checkpoint(Path(directory))
    restored = TabFORGEEmbedder.restore_from_checkpoint(checkpoint_path, device=DEVICE)
    restored_grid = restored.transform(X_test.iloc[:4])

print({"checkpoint": checkpoint_path.name, "restored_grid": restored_grid.shape})

## 10. Reproducibility and embedding selection

Keep the data split, `random_state`, checkpoint revision, extraction source, layer, and pooling fixed.
Select embeddings on validation data; report final probe metrics once on the held-out test split.